----
# SLH-DSA
----

In [1]:
import os

# Use the locally built liboqs install if available
oqs_install_path = r"C:\Users\Admin\_oqs_0.15.0"
if os.path.isdir(oqs_install_path):
    os.environ["OQS_INSTALL_PATH"] = oqs_install_path
    print("Using OQS_INSTALL_PATH:", os.environ["OQS_INSTALL_PATH"])

import oqs
import time
import numpy as np
import matplotlib.pyplot as plt
import csv
import tracemalloc
from cryptography.hazmat.primitives import hashes
from cryptography.hazmat.primitives.asymmetric import padding, rsa
from cryptography.hazmat.primitives.serialization import Encoding, NoEncryption, PrivateFormat, PublicFormat

benchmark_output_path = "slh_dsa_benchmark_results.txt"
csv_output_path = "slh_dsa_timing.csv"
rsa_benchmark_output_path = "rsa_3072_benchmark_results.txt"
rsa_csv_output_path = "rsa_3072_timing.csv"


def write_header(file_path, header_text):
    with open(file_path, "w", encoding="utf-8") as f:
        f.write(header_text + "\n")
        f.write("=" * len(header_text) + "\n")

write_header(benchmark_output_path, "SLH-DSA Benchmark Results")
write_header(rsa_benchmark_output_path, "RSA-3072 Benchmark Results")

csv_enabled = True
try:
    with open(csv_output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, delimiter=';')
        writer.writerow(["metric", "run", "time_ms", "peak_kb"])
    with open(rsa_csv_output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, delimiter=';')
        writer.writerow(["metric", "run", "time_ms", "peak_kb"])
except PermissionError:
    print(f"Warning: Cannot write to one or more CSV files (file may be open in another program). CSV logging disabled.")
    csv_enabled = False




def append_to_output(file_path, *lines):
    with open(file_path, "a", encoding="utf-8") as f:
        for line in lines:
            f.write(str(line) + "\n")


def append_to_csv(csv_path, metric, run, time_ms, peak_kb):
    if csv_enabled:
        with open(csv_path, "a", newline="", encoding="utf-8") as f:
            writer = csv.writer(f, delimiter=';')
            writer.writerow([metric, run, time_ms, peak_kb])



Using OQS_INSTALL_PATH: C:\Users\Admin\_oqs_0.15.0


c:\Users\Admin\Desktop\benchmark project\Benchmarking_Post-Quantum_Cryptographic_Algorithms\.venv\Lib\site-packages\oqs\__init__.py:1: UserWarning: liboqs version (major, minor) 0.15.0 differs from liboqs-python version 0.14.1
  from oqs.oqs import (


In [2]:
# Test SLH-DSA availability
print("Testing SLH-DSA availability...")
print("\nAvailable SLH-DSA algorithms:")
slh_algs = [alg for alg in oqs.get_enabled_sig_mechanisms() if "SLH" in alg.upper()]
for alg in slh_algs:
    print(f"  ✓ {alg}")

if not slh_algs:
    print("No SLH-DSA algorithms enabled in this OQS build.")
else:
    print("\n✓ SLH-DSA setup successful!")

Testing SLH-DSA availability...

Available SLH-DSA algorithms:
  ✓ SLH_DSA_PURE_SHA2_128S
  ✓ SLH_DSA_PURE_SHA2_128F
  ✓ SLH_DSA_PURE_SHA2_192S
  ✓ SLH_DSA_PURE_SHA2_192F
  ✓ SLH_DSA_PURE_SHA2_256S
  ✓ SLH_DSA_PURE_SHA2_256F
  ✓ SLH_DSA_PURE_SHAKE_128S
  ✓ SLH_DSA_PURE_SHAKE_128F
  ✓ SLH_DSA_PURE_SHAKE_192S
  ✓ SLH_DSA_PURE_SHAKE_192F
  ✓ SLH_DSA_PURE_SHAKE_256S
  ✓ SLH_DSA_PURE_SHAKE_256F
  ✓ SLH_DSA_SHA2_224_PREHASH_SHA2_128S
  ✓ SLH_DSA_SHA2_224_PREHASH_SHA2_128F
  ✓ SLH_DSA_SHA2_224_PREHASH_SHA2_192S
  ✓ SLH_DSA_SHA2_224_PREHASH_SHA2_192F
  ✓ SLH_DSA_SHA2_224_PREHASH_SHA2_256S
  ✓ SLH_DSA_SHA2_224_PREHASH_SHA2_256F
  ✓ SLH_DSA_SHA2_224_PREHASH_SHAKE_128S
  ✓ SLH_DSA_SHA2_224_PREHASH_SHAKE_128F
  ✓ SLH_DSA_SHA2_224_PREHASH_SHAKE_192S
  ✓ SLH_DSA_SHA2_224_PREHASH_SHAKE_192F
  ✓ SLH_DSA_SHA2_224_PREHASH_SHAKE_256S
  ✓ SLH_DSA_SHA2_224_PREHASH_SHAKE_256F
  ✓ SLH_DSA_SHA2_256_PREHASH_SHA2_128S
  ✓ SLH_DSA_SHA2_256_PREHASH_SHA2_128F
  ✓ SLH_DSA_SHA2_256_PREHASH_SHA2_192S
  ✓ SLH_DSA_SHA2

In [3]:
# Benchmark SLH-DSA keypair generation for a representative parameter set
alg = "SLH_DSA_PURE_SHA2_128F"
if alg not in slh_algs:
    raise ValueError(f"Algorithm {alg} is not enabled. Choose one of: {slh_algs}")

print(f"Benchmarking {alg} keypair generation...")
runs = 20
times_ms = []
peaks_kb = []
with oqs.Signature(alg) as sig:
    for run_index in range(runs):
        start = time.perf_counter()
        tracemalloc.start()
        _ = sig.generate_keypair()
        current, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        end = time.perf_counter()
        elapsed_ms = (end - start) * 1000
        peak_kb = peak / 1024
        times_ms.append(elapsed_ms)
        peaks_kb.append(peak_kb)
        append_to_csv(csv_output_path, "keypair_generation", run_index + 1, elapsed_ms, peak_kb)

print(f"Algorithm: {alg}")
print(f"Runs: {runs}")
print(f"Mean keypair time: {np.mean(times_ms):.3f} ms")
print(f"Min keypair time: {np.min(times_ms):.3f} ms")
print(f"Max keypair time: {np.max(times_ms):.3f} ms")
print(f"Mean peak memory: {np.mean(peaks_kb):.3f} KB")
print(f"Max peak memory: {np.max(peaks_kb):.3f} KB")
print("Times (ms):", [round(t, 3) for t in times_ms])
print("Peaks (KB):", [round(p, 3) for p in peaks_kb])

append_to_output(
    benchmark_output_path,
    "\nBenchmarking {alg} keypair generation...".format(alg=alg),
    f"Algorithm: {alg}",
    f"Runs: {runs}",
    f"Mean keypair time: {np.mean(times_ms):.3f} ms",
    f"Min keypair time: {np.min(times_ms):.3f} ms",
    f"Max keypair time: {np.max(times_ms):.3f} ms",
    f"Mean peak memory: {np.mean(peaks_kb):.3f} KB",
    f"Max peak memory: {np.max(peaks_kb):.3f} KB",
    f"Times (ms): {[round(t, 3) for t in times_ms]}",
    f"Peaks (KB): {[round(p, 3) for p in peaks_kb]}")

Benchmarking SLH_DSA_PURE_SHA2_128F keypair generation...
Algorithm: SLH_DSA_PURE_SHA2_128F
Runs: 20
Mean keypair time: 5.773 ms
Min keypair time: 4.872 ms
Max keypair time: 7.971 ms
Mean peak memory: 0.885 KB
Max peak memory: 6.787 KB
Times (ms): [5.609, 4.872, 5.511, 5.386, 5.034, 6.102, 5.036, 6.056, 5.251, 4.982, 6.211, 5.845, 6.502, 4.911, 4.901, 7.768, 4.913, 7.971, 5.813, 6.791]
Peaks (KB): [6.787, 1.206, 0.539, 0.539, 0.539, 0.539, 0.539, 0.539, 0.539, 0.539, 0.539, 0.539, 0.539, 0.539, 0.539, 0.539, 0.539, 0.539, 0.539, 0.539]


In [4]:
# Benchmark SLH-DSA signing time
message = b"Hello, world! This is a test message for SLH-DSA signing benchmark."

print(f"Benchmarking {alg} signing...")
runs = 20
sign_times_ms = []
sign_peaks_kb = []
with oqs.Signature(alg) as sig:
    pk = sig.generate_keypair()
    context = b""
    for run_index in range(runs):
        start = time.perf_counter()
        tracemalloc.start()
        signature = sig.sign_with_ctx_str(message, context)
        current, peak = tracemalloc.get_traced_memory()
        tracemalloc.stop()
        end = time.perf_counter()
        elapsed_ms = (end - start) * 1000
        peak_kb = peak / 1024
        sign_times_ms.append(elapsed_ms)
        sign_peaks_kb.append(peak_kb)
        append_to_csv(csv_output_path, "signing", run_index + 1, elapsed_ms, peak_kb)

print(f"Algorithm: {alg}")
print(f"Runs: {runs}")
print(f"Mean signing time: {np.mean(sign_times_ms):.3f} ms")
print(f"Min signing time: {np.min(sign_times_ms):.3f} ms")
print(f"Max signing time: {np.max(sign_times_ms):.3f} ms")
print(f"Mean peak memory: {np.mean(sign_peaks_kb):.3f} KB")
print(f"Max peak memory: {np.max(sign_peaks_kb):.3f} KB")
print("Signing times (ms):", [round(t, 3) for t in sign_times_ms])
print("Peaks (KB):", [round(p, 3) for p in sign_peaks_kb])

append_to_output(
    benchmark_output_path,
    "\nBenchmarking {alg} signing...".format(alg=alg),
    f"Algorithm: {alg}",
    f"Runs: {runs}",
    f"Mean signing time: {np.mean(sign_times_ms):.3f} ms",
    f"Min signing time: {np.min(sign_times_ms):.3f} ms",
    f"Max signing time: {np.max(sign_times_ms):.3f} ms",
    f"Mean peak memory: {np.mean(sign_peaks_kb):.3f} KB",
    f"Max peak memory: {np.max(sign_peaks_kb):.3f} KB",
    f"Signing times (ms): {[round(t, 3) for t in sign_times_ms]}",
    f"Peaks (KB): {[round(p, 3) for p in sign_peaks_kb]}")

Benchmarking SLH_DSA_PURE_SHA2_128F signing...
Algorithm: SLH_DSA_PURE_SHA2_128F
Runs: 20
Mean signing time: 133.702 ms
Min signing time: 113.531 ms
Max signing time: 155.074 ms
Mean peak memory: 34.506 KB
Max peak memory: 41.562 KB
Signing times (ms): [122.541, 136.294, 138.033, 133.963, 126.16, 155.074, 128.164, 127.674, 126.604, 113.531, 123.207, 150.526, 131.478, 135.736, 127.553, 142.673, 131.261, 146.744, 142.069, 134.758]
Peaks (KB): [41.562, 34.23, 34.129, 34.129, 34.129, 34.129, 34.129, 34.129, 34.129, 34.129, 34.129, 34.129, 34.129, 34.129, 34.129, 34.129, 34.129, 34.129, 34.129, 34.129]


In [5]:
# SLH-DSA Verification Benchmark
# Note: Verification is currently not working due to a possible bug in the oqs library for SLH-DSA algorithms.
# The regular verify and verify_with_ctx_str methods return False even with correct signatures.
# This may be due to the library version or implementation issues.
# Skipping verification benchmark for now.

In [6]:
# Measure SLH-DSA size metrics
print(f"Measuring {alg} size metrics...")
with oqs.Signature(alg) as sig:
    pk = sig.generate_keypair()
    sk = sig.export_secret_key()
    context = b""
    signature = sig.sign_with_ctx_str(message, context)

print(f"Algorithm: {alg}")
print(f"Public key size: {len(pk)} bytes")
print(f"Secret key size: {len(sk)} bytes")
print(f"Signature size: {len(signature)} bytes")

append_to_output(
    benchmark_output_path,
    "\nMeasuring {alg} size metrics...".format(alg=alg),
    f"Algorithm: {alg}",
    f"Public key size: {len(pk)} bytes",
    f"Secret key size: {len(sk)} bytes",

    f"Signature size: {len(signature)} bytes")

Measuring SLH_DSA_PURE_SHA2_128F size metrics...
Algorithm: SLH_DSA_PURE_SHA2_128F
Public key size: 32 bytes
Secret key size: 64 bytes
Signature size: 17088 bytes


In [7]:
# RSA-3072 comparison benchmark
rsa_message = b"Hello, world! This is a test message for RSA-3072 benchmark."
rsa_runs = 20

print("Benchmarking RSA-3072 keypair generation...")
rsa_keygen_times = []
rsa_keygen_peaks_kb = []
for run_index in range(rsa_runs):
    start = time.perf_counter()
    tracemalloc.start()
    private_key = rsa.generate_private_key(public_exponent=65537, key_size=3072)
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    end = time.perf_counter()
    elapsed_ms = (end - start) * 1000
    peak_kb = peak / 1024
    rsa_keygen_times.append(elapsed_ms)
    rsa_keygen_peaks_kb.append(peak_kb)
    append_to_csv(rsa_csv_output_path, "keypair_generation", run_index + 1, elapsed_ms, peak_kb)

print(f"Runs: {rsa_runs}")
print(f"Mean keypair time: {np.mean(rsa_keygen_times):.3f} ms")
print(f"Min keypair time: {np.min(rsa_keygen_times):.3f} ms")
print(f"Max keypair time: {np.max(rsa_keygen_times):.3f} ms")
print(f"Mean peak memory: {np.mean(rsa_keygen_peaks_kb):.3f} KB")
print(f"Max peak memory: {np.max(rsa_keygen_peaks_kb):.3f} KB")
print("Times (ms):", [round(t, 3) for t in rsa_keygen_times])
print("Peaks (KB):", [round(p, 3) for p in rsa_keygen_peaks_kb])

append_to_output(
    rsa_benchmark_output_path,
    "\nBenchmarking RSA-3072 keypair generation...",
    f"Runs: {rsa_runs}",
    f"Mean keypair time: {np.mean(rsa_keygen_times):.3f} ms",
    f"Min keypair time: {np.min(rsa_keygen_times):.3f} ms",
    f"Max keypair time: {np.max(rsa_keygen_times):.3f} ms",
    f"Mean peak memory: {np.mean(rsa_keygen_peaks_kb):.3f} KB",
    f"Max peak memory: {np.max(rsa_keygen_peaks_kb):.3f} KB",
    f"Times (ms): {[round(t, 3) for t in rsa_keygen_times]}",
    f"Peaks (KB): {[round(p, 3) for p in rsa_keygen_peaks_kb]}")

print("Benchmarking RSA-3072 signing...")
rsa_sign_times = []
rsa_sign_peaks_kb = []
public_key = private_key.public_key()
for run_index in range(rsa_runs):
    start = time.perf_counter()
    tracemalloc.start()
    signature = private_key.sign(
        rsa_message,
        padding.PSS(mgf=padding.MGF1(hashes.SHA256()), salt_length=padding.PSS.MAX_LENGTH),
        hashes.SHA256(),
    )
    current, peak = tracemalloc.get_traced_memory()
    tracemalloc.stop()
    end = time.perf_counter()
    elapsed_ms = (end - start) * 1000
    peak_kb = peak / 1024
    rsa_sign_times.append(elapsed_ms)
    rsa_sign_peaks_kb.append(peak_kb)
    append_to_csv(rsa_csv_output_path, "signing", run_index + 1, elapsed_ms, peak_kb)

print(f"Runs: {rsa_runs}")
print(f"Mean signing time: {np.mean(rsa_sign_times):.3f} ms")
print(f"Min signing time: {np.min(rsa_sign_times):.3f} ms")
print(f"Max signing time: {np.max(rsa_sign_times):.3f} ms")
print(f"Mean peak memory: {np.mean(rsa_sign_peaks_kb):.3f} KB")
print(f"Max peak memory: {np.max(rsa_sign_peaks_kb):.3f} KB")
print("Signing times (ms):", [round(t, 3) for t in rsa_sign_times])
print("Peaks (KB):", [round(p, 3) for p in rsa_sign_peaks_kb])

append_to_output(
    rsa_benchmark_output_path,
    "\nBenchmarking RSA-3072 signing...",
    f"Runs: {rsa_runs}",
    f"Mean signing time: {np.mean(rsa_sign_times):.3f} ms",
    f"Min signing time: {np.min(rsa_sign_times):.3f} ms",
    f"Max signing time: {np.max(rsa_sign_times):.3f} ms",
    f"Mean peak memory: {np.mean(rsa_sign_peaks_kb):.3f} KB",
    f"Max peak memory: {np.max(rsa_sign_peaks_kb):.3f} KB",
    f"Signing times (ms): {[round(t, 3) for t in rsa_sign_times]}",
    f"Peaks (KB): {[round(p, 3) for p in rsa_sign_peaks_kb]}")

print("Measuring RSA-3072 size metrics...")
rsa_private_bytes = private_key.private_bytes(
    encoding=Encoding.PEM,
    format=PrivateFormat.TraditionalOpenSSL,
    encryption_algorithm=NoEncryption(),
)
rsa_public_bytes = public_key.public_bytes(
    encoding=Encoding.PEM,
    format=PublicFormat.SubjectPublicKeyInfo,
)

print(f"Public key size: {len(rsa_public_bytes)} bytes")
print(f"Private key size: {len(rsa_private_bytes)} bytes")
print(f"Signature size: {len(signature)} bytes")

append_to_output(
    rsa_benchmark_output_path,
    "\nMeasuring RSA-3072 size metrics...",
    f"Public key size: {len(rsa_public_bytes)} bytes",
    f"Private key size: {len(rsa_private_bytes)} bytes",
    f"Signature size: {len(signature)} bytes")

Benchmarking RSA-3072 keypair generation...
Runs: 20
Mean keypair time: 243.673 ms
Min keypair time: 43.041 ms
Max keypair time: 450.447 ms
Mean peak memory: 0.244 KB
Max peak memory: 2.962 KB
Times (ms): [59.323, 339.631, 43.041, 82.882, 244.302, 139.097, 447.846, 128.764, 334.907, 387.502, 218.234, 148.833, 143.598, 188.768, 422.329, 193.714, 333.964, 450.447, 245.856, 320.428]
Peaks (KB): [2.962, 0.792, 0.023, 0.023, 0.023, 0.023, 0.023, 0.023, 0.023, 0.023, 0.719, 0.023, 0.023, 0.023, 0.023, 0.023, 0.023, 0.023, 0.023, 0.023]
Benchmarking RSA-3072 signing...
Runs: 20
Mean signing time: 3.931 ms
Min signing time: 2.902 ms
Max signing time: 7.974 ms
Mean peak memory: 1.395 KB
Max peak memory: 4.889 KB
Signing times (ms): [7.974, 3.362, 3.29, 4.405, 3.849, 2.977, 3.856, 3.819, 3.762, 3.755, 3.741, 3.479, 3.071, 3.186, 2.902, 4.241, 4.287, 4.632, 4.246, 3.782]
Peaks (KB): [4.889, 1.639, 1.576, 1.529, 1.482, 1.436, 1.373, 1.311, 1.264, 1.217, 1.154, 1.107, 1.061, 1.037, 1.021, 0.99, 0.9